# Upload and Download a Skill Package `@azure/ai-projects`

This notebook demonstrates uploading and downloading a skill package using the `AIProjectClient`:

1. Upload a package with `createFromFiles(...)` under a unique, sample-owned skill name.
2. Retrieve the uploaded skill with `get(...)`.
3. Download the package with `download(...)` to a folder.
4. Delete the uploaded skill.

Skills are a preview feature. In the JS SDK, you access these operations via `project.beta.skills`.

It mirrors the [`skillUploadAndDownload.ts`](./skillUploadAndDownload.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current, from the repo root: `pnpm turbo build --filter=@azure/ai-projects... --token 1`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`.
6. A skill package zip present at `samples-dev/assets/canvas-design.zip`.

Run the cells in order (top to bottom); state is shared across cells.

In [6]:
// Imports and configuration
import { AIProjectClient } from "@azure/ai-projects";
import { DefaultAzureCredential } from "@azure/identity";
import { readFileSync, writeFileSync } from "node:fs";
import path from "node:path";
import { buffer } from "node:stream/consumers";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
// Use a unique, sample-owned name so this notebook never collides with (and
// deletes) a skill that already exists in the project.
const timestamp = new Date().toISOString().replace(/[:.]/g, "-");
const skillName = `canvas-design-${timestamp}`;
const skillFilePath = path.resolve(process.cwd(), "..", "assets", "canvas-design.zip");
const downloadFolder = process.cwd();

In [7]:
// Create the AI Project client
const project = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());

In [9]:
// Upload a skill package
const packageBytes = readFileSync(skillFilePath);
const imported: any = await project.beta.skills.createFromFiles(skillName, {
  files: [
    { contents: packageBytes, contentType: "application/zip", filename: "canvas-design.zip" },
  ],
});
console.log(`Imported skill from package: ${imported.name} (${imported.skill_id})`);

Imported skill from package: canvas-design (skill_6476487ab105cb285965d40cff3ab78bcf651277)


In [10]:
// Retrieve the uploaded skill
const fetched: any = await project.beta.skills.get(imported.name);
console.log(`Fetched imported skill: ${fetched.name} (${fetched.id})`);

Fetched imported skill: canvas-design (skill_6476487ab105cb285965d40cff3ab78bcf651277)


In [12]:
// Download the skill package
const downloadPath = path.join(downloadFolder, `${fetched.name}.zip`);
const downloadResult: any = await project.beta.skills.download(fetched.name);

const content = downloadResult.readableStreamBody
  ? new Uint8Array(await buffer(downloadResult.readableStreamBody))
  : downloadResult.blobBody
    ? new Uint8Array(await (await downloadResult.blobBody).arrayBuffer())
    : undefined;

if (content) {
  writeFileSync(downloadPath, content);
  console.log(`Downloaded skill package: ${content.length} bytes`);
} else {
  console.warn("No content found in the downloaded skill package.");
}

Downloaded skill package: 2645548 bytes


In [13]:
// Delete the uploaded skill
const deleteResult: any = await project.beta.skills.delete(fetched.name);
console.log(`Deleted imported skill: ${deleteResult.name} (deleted: ${deleteResult.deleted})`);

Deleted imported skill: canvas-design (deleted: true)
